In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
import numpy as np

import pyspark
from pyspark.streaming import StreamingContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, MapType, ArrayType

In [2]:
spark = SparkSession.builder \
    .appName("Logs Transformation Code") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    ) \
    .getOrCreate()

print("Sessão Spark iniciada com sucesso!")

Sessão Spark iniciada com sucesso!


In [3]:
df = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "broker:9092") \
  .option("subscribe", "topic_log_monitoring") \
  .option("startingOffsets", "earliest") \
  .load() 

print("Executado")

Executado


In [4]:
df_string = df.selectExpr("CAST(value AS STRING)")

In [5]:
df_string.printSchema()

root
 |-- value: string (nullable = true)



In [6]:
schema_logs = StructType([ 
    StructField("insert_id", StringType(), True), 
    StructField("logName", StringType(), True), 
    StructField("severity", StringType(), True), 
    StructField("timestamp",StringType(), True),

    StructField("resource", StructType([
        StructField("type", StringType(), True),
        StructField("labels", MapType(StringType(), StringType()), True)
    ]), True),    

    StructField("protoPayload", StructType([
        StructField("@type", StringType(), True),
        StructField("serviceName", StringType(), True),
        StructField("methodName", StringType(), True),
        StructField("resourceName", StringType(), True),
        StructField("authenticationInfo", MapType(StringType(), StringType()), True),
        StructField("authorizationInfo",
            ArrayType(
                StructType([
                    StructField("resource", StringType(), True),
                    StructField("permission", StringType(), True),
                    StructField("granted", StringType(), True),
                    StructField(
                        "resourceAttributes",
                        MapType(StringType(), StringType()),
                        True
                    )
                ])
            ),
            True
        ),
    ])),
])

In [9]:
def bronze_table():
    df_parsed = df_string.withColumn("jsonData", sf.from_json(sf.col("value"), schema_logs)).select("jsonData.*", "value")

    df_bronze = df_parsed.withColumn(
        "metadata", 
        sf.get_json_object(sf.col("value"), "$.protoPayload.metadata") 
    )

    df_bronze = df_bronze.drop("value")

In [11]:
bronze_table()

In [12]:
query = (
    df_bronze.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .start()
)

# spark.streams.awaitAnyTermination()